In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import MinMaxScaler

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score

In [3]:
data = pd.read_csv("../Raisin_Dataset.csv")

In [4]:
x = data.drop("Class",axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y
)

pipelineSvc = Pipeline([
    ('scaler',MinMaxScaler()),
    ('model',SVC(kernel='rbf'))
])
pipelineSvc.fit(X_train, y_train)
y_predSvc = pipelineSvc.predict(X_test)


pipelineknn = Pipeline([
    ('scaler',MinMaxScaler()),
    ('model',KNeighborsClassifier(n_neighbors=5))
])
pipelineknn.fit(X_train, y_train)
y_predknn = pipelineknn.predict(X_test)


pipelineDt = Pipeline([
    ('model', DecisionTreeClassifier(random_state=42))
])

pipelineDt.fit(X_train, y_train)

y_predDt = pipelineDt.predict(X_test)

scoresSvc = cross_val_score(pipelineSvc, X_train, y_train, cv=5, scoring="accuracy")
scoresDt = cross_val_score(pipelineDt, X_train, y_train, cv=5, scoring="accuracy")
scoresknn = cross_val_score(pipelineknn, X_train, y_train, cv=5, scoring="accuracy")

def analyser_stabilite(nom, scores):
    print(f"\n--- {nom} ---")
    print("5 scores :", scores)
    print("Moyenne (μ) :", scores.mean())
    print("Écart-type (σ) :", scores.std())
    print("Minimum :", scores.min())
    print("Maximum :", scores.max())
    
analyser_stabilite("SVC", scoresSvc)
analyser_stabilite("Decision Tree", scoresDt)
analyser_stabilite("KNN", scoresknn)


--- SVC ---
5 scores : [0.77777778 0.90277778 0.85416667 0.86111111 0.93055556]
Moyenne (μ) : 0.8652777777777778
Écart-type (σ) : 0.05189317136741502
Minimum : 0.7777777777777778
Maximum : 0.9305555555555556

--- Decision Tree ---
5 scores : [0.77777778 0.88194444 0.73611111 0.83333333 0.84027778]
Moyenne (μ) : 0.8138888888888889
Écart-type (σ) : 0.051106581924928035
Minimum : 0.7361111111111112
Maximum : 0.8819444444444444

--- KNN ---
5 scores : [0.8125     0.89583333 0.83333333 0.83333333 0.89583333]
Moyenne (μ) : 0.8541666666666667
Écart-type (σ) : 0.034860834438919824
Minimum : 0.8125
Maximum : 0.8958333333333334


In [5]:
from sklearn.model_selection import GridSearchCV

In [6]:
param_grid = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    estimator=pipelineSvc, #notre model ou pipeline
    param_grid=param_grid, #hyperparameter qui nous veulons tester
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train,y_train)

print("Best Parameters:", grid_search.best_params_)

print("Best Accuracy:", grid_search.best_score_)

Best Parameters: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best Accuracy: 0.8652777777777778


In [7]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.8833333333333333


In [8]:
import joblib
joblib.dump(best_model, "raisin_best_pipeline.pkl")

['raisin_best_pipeline.pkl']